# Séance 7 — CNN : architecture de base

**Objectifs.**
- Comprendre ce que fait concrètement une convolution (filtre, stride, padding) et un pooling.
- Observer des feature maps produites par un filtre convolutif.
- Coder une architecture simple type LeNet / mini-VGG en `nn.Module`.
- Réutiliser le `Trainer` de `training_toolbox.py` **sans aucune modification** : côté
  outillage, rien de nouveau cette séance, tout se joue dans la définition du modèle.

On travaille sur **Fashion-MNIST** (images 28x28 en niveaux de gris, 10 classes de
vêtements), plus intéressant que MNIST pour un CNN car les classes ne se distinguent pas
uniquement par des traits simples.


In [ ]:
# !pip install -q torch torchvision matplotlib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split

import torchvision
from torchvision import transforms

import matplotlib.pyplot as plt

from training_toolbox import Trainer, accuracy

torch.manual_seed(0)


## Partie 1 — Données : Fashion-MNIST

Chargement via `torchvision.datasets`. On garde le format image `(1, 28, 28)` (1 canal,
niveaux de gris) : c'est le format attendu par les couches `nn.Conv2d` (contrairement au MLP
des séances précédentes, on ne "aplatit" plus l'image en vecteur).


In [ ]:
CLASSES = [
    "T-shirt/top", "Pantalon", "Pull", "Robe", "Manteau",
    "Sandale", "Chemise", "Basket", "Sac", "Bottine",
]

transform = transforms.ToTensor()  # convertit en tenseur (C, H, W), valeurs dans [0, 1]

full_train = torchvision.datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)
test_set = torchvision.datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transform
)

# On garde un petit set de validation à part du train
n_val = 5000
n_train = len(full_train) - n_val
train_set, val_set = random_split(full_train, [n_train, n_val])

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
val_loader = DataLoader(val_set, batch_size=256)
test_loader = DataLoader(test_set, batch_size=256)

x_batch, y_batch = next(iter(train_loader))
print("Shape d'un batch d'images :", x_batch.shape)  # (batch, 1, 28, 28)
print("Shape des labels :", y_batch.shape)


In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for ax, img, label in zip(axes.ravel(), x_batch, y_batch):
    ax.imshow(img.squeeze(0), cmap="gray")
    ax.set_title(CLASSES[label.item()], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()


## Partie 2 — Ce que fait une convolution

Avant de coder un CNN entraînable, regardons ce que fait *un seul* filtre convolutif, à la
main, avec des poids fixés (pas d'apprentissage ici, juste de la visualisation). On applique
un filtre détecteur de contours verticaux à une image.

Ceci illustre trois notions vues en cours :
- le **filtre** (noyau de convolution, ici 3x3) ;
- le **stride** (pas de déplacement du filtre) ;
- le **padding** (ce qu'on fait faire au filtre sur les bords de l'image).


In [ ]:
# Filtre détecteur de contours verticaux (Sobel), fixé à la main
sobel_x = torch.tensor([
    [-1., 0., 1.],
    [-2., 0., 2.],
    [-1., 0., 1.],
]).view(1, 1, 3, 3)  # (out_channels, in_channels, kH, kW) : le format attendu par F.conv2d

image = x_batch[0:1]  # (1, 1, 28, 28) : on garde la dimension batch

feature_map_valid = F.conv2d(image, sobel_x, stride=1, padding=0)
feature_map_same = F.conv2d(image, sobel_x, stride=1, padding=1)
feature_map_stride2 = F.conv2d(image, sobel_x, stride=2, padding=1)

print("Image d'entrée       :", image.shape)
print("padding=0 (stride 1) :", feature_map_valid.shape)
print("padding=1 (stride 1) :", feature_map_same.shape)
print("padding=1, stride=2  :", feature_map_stride2.shape)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(image[0, 0], cmap="gray"); axes[0].set_title("Image d'origine")
axes[1].imshow(feature_map_valid[0, 0].detach(), cmap="gray"); axes[1].set_title("padding=0")
axes[2].imshow(feature_map_same[0, 0].detach(), cmap="gray"); axes[2].set_title("padding=1")
axes[3].imshow(feature_map_stride2[0, 0].detach(), cmap="gray"); axes[3].set_title("padding=1, stride=2")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


**Questions.**

- Pourquoi la sortie avec `padding=0` est-elle plus petite que l'image d'entrée ? De combien
  de pixels exactement, et pourquoi (en fonction de la taille du filtre) ?
- Que fait le `padding=1` ici ? Pourquoi restaure-t-il (à peu près) la taille d'entrée pour un
  filtre 3x3 ?
- Quel est l'effet du `stride=2` sur la taille de la sortie ? À quoi cela correspond-il, en
  termes de calcul, par rapport à un pooling ?
- Ce filtre est un détecteur de contours *verticaux* : sur l'image affichée, où voyez-vous
  la réponse la plus forte (en valeur absolue) ?


## Partie 3 — Une architecture mini-VGG en `nn.Module`

Dans un vrai CNN, on **n'écrit pas** les filtres à la main : ce sont des paramètres appris
par descente de gradient, exactement comme les poids d'un MLP. On empile plusieurs blocs
`Conv2d -> ReLU`, entrecoupés de `MaxPool2d` pour réduire progressivement la résolution
spatiale tout en augmentant le nombre de canaux (l'idée classique "mini-VGG").

**Architecture à coder :**

- **Bloc 1** : `Conv2d(1, 32, kernel_size=3, padding=1)` → `ReLU` →
  `Conv2d(32, 32, kernel_size=3, padding=1)` → `ReLU` → `MaxPool2d(2)`
- **Bloc 2** : `Conv2d(32, 64, kernel_size=3, padding=1)` → `ReLU` →
  `Conv2d(64, 64, kernel_size=3, padding=1)` → `ReLU` → `MaxPool2d(2)`
- **Classifieur** : `Flatten` → `Linear(64 * 7 * 7, 128)` → `ReLU` → `Linear(128, 10)`

**À vous de jouer : complétez les `TODO` ci-dessous.**


In [ ]:
class MiniVGG(nn.Module):
    """CNN type mini-VGG : deux blocs (conv, conv, pool) puis un classifieur dense.

    Après le bloc 1, une image (1, 28, 28) devient (32, 14, 14) (pooling /2).
    Après le bloc 2, elle devient (64, 7, 7) (nouveau pooling /2).
    """

    def __init__(self, n_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = MiniVGG(n_classes=10)

# Vérification rapide des dimensions : ne doit pas planter, et doit renvoyer (batch, 10)
with torch.no_grad():
    out = model(x_batch)
print("Sortie du modèle :", out.shape)

n_params = sum(p.numel() for p in model.parameters())
print(f"Nombre total de paramètres : {n_params:,}")

**Questions.**

- Combien de paramètres compte la première couche convolutive (`Conv2d(1, 32, 3)`) ? Et la
  première couche `Linear` du classifieur ? Que remarquez-vous sur leurs ordres de grandeur
  respectifs ?
- Pourquoi la dimension d'entrée du premier `Linear` est-elle `64 * 7 * 7` et pas, disons,
  `64 * 28 * 28` ?


## Partie 4 — Entraînement avec le `Trainer`

Rien de nouveau ici : on réutilise exactement le `Trainer` des séances précédentes, avec
`CrossEntropyLoss` (classification multiclasse) et la métrique `accuracy` déjà fournie par
`training_toolbox.py`.


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

trainer = Trainer(model, optimizer, loss_fn, metrics={"acc": accuracy})
history = trainer.fit(train_loader, val_loader, epochs=5)


In [ ]:
def plot_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"], label="val")
    axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
    axes[1].plot(history["train_acc"], label="train")
    axes[1].plot(history["val_acc"], label="val")
    axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_history(history)

test_stats = trainer.evaluate(test_loader)
print("Performance sur le test set :", test_stats)


## Partie 5 (bonus) — Visualiser les filtres appris

Les poids de la première couche convolutive sont directement interprétables comme des
petites images 3x3 : on peut les afficher pour voir quel type de motif chaque filtre détecte
(contours, coins, dégradés...).


In [ ]:
first_conv = model.features[0]  # le tout premier Conv2d(1, 32, 3, padding=1)
weights = first_conv.weight.detach().cpu()  # shape (32, 1, 3, 3)

fig, axes = plt.subplots(4, 8, figsize=(10, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(weights[i, 0], cmap="gray")
    ax.axis("off")
plt.suptitle("Filtres 3x3 appris par la première couche convolutive")
plt.tight_layout()
plt.show()


**Questions de compréhension.**

- Ces filtres ressemblent-ils à des détecteurs de contours comme celui de la Partie 2 ? Sont-
  ils tous "interprétables" à l'œil nu ?
- Le pooling (`MaxPool2d`) ne contient aucun paramètre appris. À quoi sert-il alors ? Quel
  est l'effet indésirable qu'il pourrait avoir si on l'utilisait de façon trop agressive
  (trop de pooling, trop tôt) ?
- Un MLP entièrement connecté appliqué directement sur l'image 28x28 aplatie aurait, dès sa
  première couche, un nombre de paramètres bien supérieur à celui de `Conv2d(1, 32, 3)`.
  Pourquoi ? Qu'est-ce que la convolution exploite dans la structure des images pour réduire
  ce nombre de paramètres (indice : partage de poids, connectivité locale) ?

## Pour aller plus loin (optionnel)

- Remplacer un `MaxPool2d(2)` par un `AvgPool2d(2)` : quel effet sur les performances ?
- Remplacer le pooling par un stride de 2 directement dans la convolution (`Conv2d(...,
  stride=2)`) : est-ce équivalent ? Qu'est-ce qui diffère ?
- Ce mini-VGG et son entraînement serviront de point de comparaison à la séance 8 (blocs
  résiduels, batch normalization).
